In [ ]:
# fire_analysis_3class_only.py
# Burn Severity Analysis using 3 classes (Low, Medium, High) + Enhanced Regrowth
# Анализ на степента на изгаряне с 3 класа + подобрен растеж
# Corrected CRS: automatically selects UTM 34N or UTM 35N for Bulgaria
# Коригирана координатна система: автоматично избира UTM 34N или UTM 35N за България
# Feature: exports all fire statistics to a single CSV file

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio import features
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from datetime import datetime, timedelta
from IPython.display import display, Image as IPImage, HTML
from rasterio.warp import transform
from pyproj import CRS, Transformer
import re

# =============================================================================
# Configuration / Конфигурация
# =============================================================================
PREVIEW_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\preview'
NBR_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\nbr_analysis'
VECTOR_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\best_classifier'
os.makedirs(NBR_DIR, exist_ok=True)

BURN_CLASSES_3CLASS = {
    0: {'name': 'Low Severity', 'color': (255, 255, 0), 'dNBR_range': '0.1 - 0.2'},
    1: {'name': 'Medium Severity', 'color': (255, 165, 0), 'dNBR_range': '0.2 - 0.3'},
    2: {'name': 'High Severity', 'color': (255, 0, 0), 'dNBR_range': '>= 0.3'}
}

# Extended severity classes including unburned and enhanced regrowth
SEVERITY_NAMES = {0: 'Enhanced Regrowth', 1: 'Unburned', 2: 'Low Severity',
                  3: 'Medium Severity', 4: 'High Severity'}

LANDCOVER_CLASSES = {
    2: {'name': 'Water', 'color': (0, 0, 255), 'description': 'Water bodies', 'include_in_vulnerability': False},
    3: {'name': 'Bare and Urban Territories', 'color': (0, 0, 0), 'description': 'Barren land and urban areas', 'include_in_vulnerability': False},
    4: {'name': 'Field/Agriculture', 'color': (255, 255, 100), 'description': 'Agricultural fields', 'include_in_vulnerability': True},
    5: {'name': 'Coniferous Forest', 'color': (34, 139, 34), 'description': 'Coniferous forest areas', 'include_in_vulnerability': True},
    6: {'name': 'Deciduous Forest', 'color': (0, 150, 0), 'description': 'Deciduous forest areas', 'include_in_vulnerability': True}
}

EXCLUDED_CLASSES = [2, 3]
VULNERABLE_CLASSES = [4, 5, 6]

def extract_fire_id_from_filename(filename):
    """
    Extract fire ID from filename.
    Извлича идентификатор на пожар от името на файла.
    """
    try:
        filename_lower = filename.lower()
        parts = filename.split('_')
        
        for i, part in enumerate(parts):
            if part.lower() == 'fire' and i + 1 < len(parts):
                next_part = parts[i + 1]
                if next_part.isdigit():
                    return f"{int(next_part):03d}"
        
        if 'fire_' in filename_lower:
            matches = re.findall(r'fire[_-](\d+)', filename_lower)
            if matches:
                return f"{int(matches[0]):03d}"
        
        for part in parts:
            clean_part = part.split('.')[0]
            if clean_part.isdigit() and 1 <= len(clean_part) <= 3:
                return f"{int(clean_part):03d}"
                
    except Exception:
        pass
    return None

def extract_dates_from_filename(filename):
    """
    Extract date (YYYYMMDD) from filename.
    Извлича дата (ГГГГММДД) от името на файла.
    """
    match = re.search(r'(\d{8})', filename)
    if match:
        date_str = match.group(1)
        try:
            return datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')
        except:
            pass
    return None

def find_and_group_image_files():
    """
    Find all image files and group them by fire ID.
    Намира всички файлове с изображения и ги групира по идентификатор на пожар.
    """
    if not os.path.exists(PREVIEW_DIR):
        print(f"Preview directory not found: {PREVIEW_DIR}")
        return {}, {}
    
    all_files = [f for f in os.listdir(PREVIEW_DIR) if f.endswith(('.tif', '.tiff'))]
    print(f"Found {len(all_files)} TIFF files total")
    
    fire_groups = {}
    for file in all_files:
        fire_id = extract_fire_id_from_filename(file)
        if fire_id:
            if fire_id not in fire_groups:
                fire_groups[fire_id] = {'before': [], 'after': []}
            
            if 'before' in file.lower():
                fire_groups[fire_id]['before'].append(file)
            elif 'after' in file.lower():
                fire_groups[fire_id]['after'].append(file)
    
    print(f"Grouped into {len(fire_groups)} fire locations")
    return all_files, fire_groups

def check_existing_outputs(before_file, after_file):
    """
    Check if output files already exist for a fire pair.
    Проверява дали изходните файлове вече съществуват за двойка пожар.
    """
    base_name = f"{os.path.splitext(before_file)[0]}_vs_{os.path.splitext(after_file)[0]}"
    
    dashboard_path = os.path.join(NBR_DIR, f"{base_name}_dashboard.png")
    nbr_path = os.path.join(NBR_DIR, f"{base_name}_nbr_after.tif")
    dnbr_path = os.path.join(NBR_DIR, f"{base_name}_dnbr.tif")
    severity_path = os.path.join(NBR_DIR, f"{base_name}_burn_severity_3class.tif")
    
    files_exist = {
        'dashboard': os.path.exists(dashboard_path),
        'nbr_after': os.path.exists(nbr_path),
        'dnbr': os.path.exists(dnbr_path),
        'severity': os.path.exists(severity_path),
        'all': all([os.path.exists(dashboard_path), os.path.exists(nbr_path), 
                   os.path.exists(dnbr_path), os.path.exists(severity_path)])
    }
    
    return files_exist, dashboard_path

def display_dashboard(dashboard_path, fire_id):
    """
    Display dashboard image in notebook.
    Показва изображението на дашборда в Jupyter notebook.
    """
    if os.path.exists(dashboard_path):
        print(f"\n{'='*60}")
        print(f"🔥 FIRE {fire_id} - VISUALIZING EXISTING DASHBOARD")
        print(f"{'='*60}")
        display(HTML(f"<h3>Fire {fire_id} - Burn Severity Analysis Dashboard</h3>"))
        display(IPImage(filename=dashboard_path, width=1200))
        return True
    else:
        print(f"❌ Dashboard not found for Fire {fire_id}: {dashboard_path}")
        return False

def identify_sentinel2_bands_for_nbr(bands):
    """
    Identify Sentinel-2 bands needed for NBR calculation.
    Определя каналите на Sentinel-2, необходими за изчисляване на NBR.
    """
    band_info = {}
    
    if len(bands) >= 13:
        band_info = {'nir': bands[7], 'swir2': bands[11]}
        print("  Using B08 for NIR, B12 for SWIR")
    elif len(bands) >= 12:
        band_info = {'nir': bands[7], 'swir2': bands[11]}
        print("  Using B8 for NIR, B12 for SWIR")
    elif len(bands) >= 10:
        band_info = {'nir': bands[7], 'swir2': bands[10] if len(bands) > 10 else bands[9]}
        print("  Using bands 8 (NIR) and 11/10 (SWIR)")
    else:
        band_means = [np.mean(band) for band in bands]
        nir_idx = np.argmax(band_means)
        remaining = [i for i in range(len(bands)) if i != nir_idx]
        if remaining:
            swir_idx = remaining[np.argmin([band_means[i] for i in remaining])]
            band_info = {'nir': bands[nir_idx], 'swir2': bands[swir_idx]}
            print("  Identified by reflectance")
    
    return band_info

def calculate_nbr(nir, swir2):
    """
    Calculate Normalized Burn Ratio.
    Изчислява Нормализиран индекс на изгаряне (NBR).
    """
    nir = nir.astype(np.float32)
    swir2 = swir2.astype(np.float32)
    
    denominator = nir + swir2
    valid_mask = denominator > 0
    
    nbr = np.zeros_like(nir, dtype=np.float32)
    nbr[valid_mask] = (nir[valid_mask] - swir2[valid_mask]) / denominator[valid_mask]
    nbr[~valid_mask] = 0
    nbr = np.clip(nbr, -1, 1)
    
    return nbr

def calculate_dnbr(nbr_before, nbr_after):
    """
    Calculate differenced NBR (dNBR = NBR_before - NBR_after).
    Изчислява разликата в NBR (dNBR = NBR_преди - NBR_след).
    """
    return nbr_before - nbr_after

def classify_burn_severity_extended(dNBR):
    """
    Classify into 5 classes: 0=Enhanced Regrowth, 1=Unburned, 2=Low, 3=Moderate, 4=High.
    Класифицира в 5 класа: 0=Подобрен растеж, 1=Неизгоряло, 2=Ниска, 3=Средна, 4=Висока.
    """
    classification = np.zeros(dNBR.shape, dtype=np.uint8)
    
    # Enhanced Regrowth (dNBR < -0.1)
    classification[dNBR < -0.1] = 0
    # Unburned (-0.1 <= dNBR < 0.1)
    classification[(dNBR >= -0.1) & (dNBR < 0.1)] = 1
    # Low Severity (0.1 <= dNBR < 0.2)
    classification[(dNBR >= 0.1) & (dNBR < 0.2)] = 2
    # Medium Severity (0.2 <= dNBR < 0.3)
    classification[(dNBR >= 0.2) & (dNBR < 0.3)] = 3
    # High Severity (dNBR >= 0.3)
    classification[dNBR >= 0.3] = 4
    
    return classification

def classify_burn_severity_3class(dNBR):
    """
    Classify burn severity into 3 classes based on dNBR thresholds (for dashboard).
    Класифицира степента на изгаряне в 3 класа според праговете на dNBR (за дашборд).
    """
    classification = np.zeros(dNBR.shape, dtype=np.uint8)
    
    classification[(dNBR >= 0.1) & (dNBR < 0.2)] = 0
    classification[(dNBR >= 0.2) & (dNBR < 0.3)] = 1
    classification[dNBR >= 0.3] = 2
    
    return classification

def find_landcover_geojson_file(fire_id):
    """
    Find land cover GeoJSON file for a fire.
    Намира GeoJSON файл с класификация на земното покритие за даден пожар.
    """
    if not os.path.exists(VECTOR_DIR):
        return None
    
    for file in os.listdir(VECTOR_DIR):
        if file.endswith('.geojson') and (fire_id in file or str(int(fire_id)) in file):
            return os.path.join(VECTOR_DIR, file)
    
    return None

def load_landcover_geojson(geojson_path, reference_transform, reference_shape):
    """
    Load and rasterize land cover GeoJSON.
    Зарежда и растеризира GeoJSON с класификация на земното покритие.
    """
    try:
        gdf = gpd.read_file(geojson_path)
        if gdf.empty:
            return None
        
        if 'class_id' not in gdf.columns:
            possible_cols = ['class', 'Class', 'CLASS', 'id', 'ID']
            for col in possible_cols:
                if col in gdf.columns:
                    gdf = gdf.rename(columns={col: 'class_id'})
                    break
        
        height, width = reference_shape
        landcover_raster = np.zeros((height, width), dtype=np.uint8)
        
        for class_id in gdf['class_id'].unique():
            class_gdf = gdf[gdf['class_id'] == class_id]
            if not class_gdf.empty:
                shapes = [(geom, class_id) for geom in class_gdf.geometry]
                rasterized = features.rasterize(
                    shapes, out_shape=(height, width),
                    transform=reference_transform, fill=0, dtype=np.uint8
                )
                landcover_raster[rasterized > 0] = class_id
        
        return landcover_raster
    except Exception as e:
        print(f"  Error loading GeoJSON: {e}")
        return None

def create_exclusion_mask(landcover_raster):
    """
    Create mask for water and barren/urban areas (to be excluded).
    Създава маска за водни и безплодни/урбанизирани територии (за изключване).
    """
    if landcover_raster is None:
        return None
    
    excluded_mask = np.zeros_like(landcover_raster, dtype=bool)
    excluded_mask[landcover_raster == 2] = True  # Water
    excluded_mask[landcover_raster == 3] = True  # Barren/Urban
    
    return excluded_mask

def calculate_overall_statistics(burn_severity, dNBR, excluded_mask=None):
    """
    Calculate overall statistics including unburned areas (for dashboard).
    """
    if excluded_mask is not None:
        valid_mask = ~excluded_mask
        severity_valid = burn_severity[valid_mask]
        dnbr_valid = dNBR[valid_mask]
        total_valid = np.sum(valid_mask)
    else:
        severity_valid = burn_severity.flatten()
        dnbr_valid = dNBR.flatten()
        total_valid = len(severity_valid)
    
    unburned_pixels = np.sum(dnbr_valid < 0.1)
    unburned_pct = (unburned_pixels / total_valid) * 100 if total_valid > 0 else 0
    
    burned_stats = {}
    for class_id in range(3):
        pixels = np.sum((severity_valid == class_id) & (dnbr_valid >= 0.1))
        pct_of_total = (pixels / total_valid) * 100 if total_valid > 0 else 0
        pct_of_burned = (pixels / (total_valid - unburned_pixels)) * 100 if (total_valid - unburned_pixels) > 0 else 0
        burned_stats[class_id] = {
            'name': BURN_CLASSES_3CLASS[class_id]['name'],
            'pixels': pixels,
            'pct_of_total': pct_of_total,
            'pct_of_burned': pct_of_burned,
            'color': BURN_CLASSES_3CLASS[class_id]['color']
        }
    
    total_burned_pixels = total_valid - unburned_pixels
    total_burned_pct = (total_burned_pixels / total_valid) * 100 if total_valid > 0 else 0
    
    return {
        'total_valid': total_valid,
        'unburned_pixels': unburned_pixels,
        'unburned_pct': unburned_pct,
        'total_burned_pixels': total_burned_pixels,
        'total_burned_pct': total_burned_pct,
        'burned_stats': burned_stats
    }

def calculate_extended_statistics(extended_severity, landcover_raster, excluded_mask):
    """
    Compute all detailed statistics for CSV export.
    Изчислява всички подробни статистики за CSV експорт.
    """
    stats = {}
    
    # Mask for valid pixels (non-excluded)
    if excluded_mask is not None:
        valid_mask = ~excluded_mask
    else:
        valid_mask = np.ones(extended_severity.shape, dtype=bool)
    
    valid_ext = extended_severity[valid_mask]
    total_valid = len(valid_ext)
    excluded_pixels = extended_severity.size - total_valid
    
    # Overall pixel counts per severity class
    counts = {code: np.sum(valid_ext == code) for code in range(5)}
    total_burned = counts[2] + counts[3] + counts[4]  # low+med+high
    unburned = counts[1]
    regrowth = counts[0]
    
    stats['Total_Valid_Pixels'] = total_valid
    stats['Excluded_Pixels'] = excluded_pixels
    stats['Excluded_Percentage'] = (excluded_pixels / (total_valid + excluded_pixels)) * 100 if (total_valid + excluded_pixels) > 0 else 0
    stats['Total_Burned_Pixels'] = total_burned
    stats['Total_Burned_%'] = (total_burned / total_valid) * 100 if total_valid > 0 else 0
    stats['Enhanced_Regrowth_%'] = (regrowth / total_valid) * 100 if total_valid > 0 else 0
    stats['Unburned_%'] = (unburned / total_valid) * 100 if total_valid > 0 else 0
    stats['Low_Severity_%'] = (counts[2] / total_burned) * 100 if total_burned > 0 else 0
    stats['Moderate_Severity_%'] = (counts[3] / total_burned) * 100 if total_burned > 0 else 0
    stats['High_Severity_%'] = (counts[4] / total_burned) * 100 if total_burned > 0 else 0
    
    # Most common severity among burned
    burned_counts = {2: counts[2], 3: counts[3], 4: counts[4]}
    max_class = max(burned_counts, key=burned_counts.get)
    severity_names = {2: 'Low Severity', 3: 'Moderate Severity', 4: 'High Severity'}
    stats['Most_Common_Severity'] = severity_names[max_class]
    
    # Fire Impact Assessment
    total_burned_pct = stats['Total_Burned_%']
    if total_burned_pct > 50:
        stats['Fire_Impact_Assessment'] = 'EXTENSIVE (>50%)'
    elif total_burned_pct > 10:
        stats['Fire_Impact_Assessment'] = 'MODERATE (10-50%)'
    else:
        stats['Fire_Impact_Assessment'] = 'MINOR (<10%)'
    
    # Damage Assessment based on high severity % of burned
    high_pct = stats['High_Severity_%']
    if high_pct > 20:
        stats['Damage_Assessment'] = 'CATASTROPHIC (>20% high severity)'
    elif high_pct > 10:
        stats['Damage_Assessment'] = 'SIGNIFICANT (10-20% high severity)'
    else:
        stats['Damage_Assessment'] = 'MINOR (<10% high severity)'
    
    # Pre/Post NBR means (will be added later from the main processing)
    # placeholder
    stats['PreFire_NBR_Mean'] = None
    stats['PostFire_NBR_Mean'] = None
    stats['dNBR_Mean'] = None
    
    # Per-landcover class analysis
    lc_stats = {}
    for class_id in LANDCOVER_CLASSES:
        if landcover_raster is None:
            # No landcover data
            lc_stats[class_id] = {'present': False, 'pixels': 0}
            continue
        
        class_mask = (landcover_raster == class_id)
        class_pixels = np.sum(class_mask)
        if class_pixels == 0:
            lc_stats[class_id] = {'present': False, 'pixels': 0}
            continue
        
        # Pixels of this class in valid area
        class_valid_mask = class_mask & valid_mask
        class_valid_pixels = np.sum(class_valid_mask)
        if class_valid_pixels == 0:
            lc_stats[class_id] = {'present': False, 'pixels': 0}
            continue
        
        class_severity = extended_severity[class_valid_mask]
        class_counts = {code: np.sum(class_severity == code) for code in range(5)}
        total_class = class_valid_pixels
        
        # Percentages relative to the class area
        regrowth_pct = (class_counts[0] / total_class) * 100
        unburned_pct = (class_counts[1] / total_class) * 100
        low_pct = (class_counts[2] / total_class) * 100
        mod_pct = (class_counts[3] / total_class) * 100
        high_pct = (class_counts[4] / total_class) * 100
        moderate_high_pct = ((class_counts[3] + class_counts[4]) / total_class) * 100
        low_mod_high_pct = ((class_counts[2] + class_counts[3] + class_counts[4]) / total_class) * 100
        
        # Vulnerability Index: average severity class (2=Low,3=Moderate,4=High) weighted by burned area
        burned_pixels = class_counts[2] + class_counts[3] + class_counts[4]
        if burned_pixels > 0:
            vuln_index = (class_counts[2]*1 + class_counts[3]*2 + class_counts[4]*3) / burned_pixels
        else:
            vuln_index = 0.0
        
        lc_stats[class_id] = {
            'present': True,
            'pixels': class_pixels,
            'percent_of_total': (class_pixels / (total_valid + excluded_pixels)) * 100,
            'regrowth_pct': regrowth_pct,
            'unburned_pct': unburned_pct,
            'low_pct': low_pct,
            'moderate_pct': mod_pct,
            'high_pct': high_pct,
            'moderate_high_pct': moderate_high_pct,
            'low_mod_high_pct': low_mod_high_pct,
            'vulnerability_index': vuln_index
        }
    
    stats['landcover_stats'] = lc_stats
    
    # Determine most vulnerable classes for each severity
    if landcover_raster is not None:
        vuln_classes = [c for c in VULNERABLE_CLASSES if lc_stats.get(c, {}).get('present', False)]
        if vuln_classes:
            # Most vulnerable to high severity (based on % of burned area that is high)
            high_candidates = [(c, lc_stats[c]['high_pct']) for c in vuln_classes if lc_stats[c]['low_mod_high_pct'] > 0]
            if high_candidates:
                most_high = max(high_candidates, key=lambda x: x[1])
                stats['Most_Vulnerable_HighSeverity'] = LANDCOVER_CLASSES[most_high[0]]['name']
                stats['Most_Vulnerable_HighSeverity_%'] = most_high[1]
            else:
                stats['Most_Vulnerable_HighSeverity'] = 'N/A'
                stats['Most_Vulnerable_HighSeverity_%'] = np.nan
                
            # Most vulnerable to moderate severity
            mod_candidates = [(c, lc_stats[c]['moderate_pct']) for c in vuln_classes if lc_stats[c]['low_mod_high_pct'] > 0]
            if mod_candidates:
                most_mod = max(mod_candidates, key=lambda x: x[1])
                stats['Most_Vulnerable_ModerateSeverity'] = LANDCOVER_CLASSES[most_mod[0]]['name']
                stats['Most_Vulnerable_ModerateSeverity_%'] = most_mod[1]
            else:
                stats['Most_Vulnerable_ModerateSeverity'] = 'N/A'
                stats['Most_Vulnerable_ModerateSeverity_%'] = np.nan
                
            # Most vulnerable to low severity
            low_candidates = [(c, lc_stats[c]['low_pct']) for c in vuln_classes if lc_stats[c]['low_mod_high_pct'] > 0]
            if low_candidates:
                most_low = max(low_candidates, key=lambda x: x[1])
                stats['Most_Vulnerable_LowSeverity'] = LANDCOVER_CLASSES[most_low[0]]['name']
                stats['Most_Vulnerable_LowSeverity_%'] = most_low[1]
            else:
                stats['Most_Vulnerable_LowSeverity'] = 'N/A'
                stats['Most_Vulnerable_LowSeverity_%'] = np.nan
        else:
            stats['Most_Vulnerable_HighSeverity'] = 'N/A'
            stats['Most_Vulnerable_HighSeverity_%'] = np.nan
            stats['Most_Vulnerable_ModerateSeverity'] = 'N/A'
            stats['Most_Vulnerable_ModerateSeverity_%'] = np.nan
            stats['Most_Vulnerable_LowSeverity'] = 'N/A'
            stats['Most_Vulnerable_LowSeverity_%'] = np.nan
    else:
        stats['Most_Vulnerable_HighSeverity'] = 'N/A'
        stats['Most_Vulnerable_HighSeverity_%'] = np.nan
        stats['Most_Vulnerable_ModerateSeverity'] = 'N/A'
        stats['Most_Vulnerable_ModerateSeverity_%'] = np.nan
        stats['Most_Vulnerable_LowSeverity'] = 'N/A'
        stats['Most_Vulnerable_LowSeverity_%'] = np.nan
    
    # Dominant landcover overall
    if landcover_raster is not None:
        valid_lc = landcover_raster[valid_mask]
        unique, counts_lc = np.unique(valid_lc, return_counts=True)
        # Exclude 0 (no data)
        lc_area = [(cl, cnt) for cl, cnt in zip(unique, counts_lc) if cl in LANDCOVER_CLASSES]
        if lc_area:
            dominant_cl, dominant_cnt = max(lc_area, key=lambda x: x[1])
            stats['Dominant_LandCover'] = LANDCOVER_CLASSES[dominant_cl]['name']
            stats['Dominant_LandCover_%'] = (dominant_cnt / total_valid) * 100 if total_valid > 0 else 0
        else:
            stats['Dominant_LandCover'] = 'N/A'
            stats['Dominant_LandCover_%'] = np.nan
    else:
        stats['Dominant_LandCover'] = 'N/A'
        stats['Dominant_LandCover_%'] = np.nan
    
    stats['Has_LandCover_Data'] = 'Yes' if landcover_raster is not None else 'No'
    
    return stats

def export_tif_files(nbr_after, dNBR, burn_severity, transform, before_file, after_file, output_dir, crs, excluded_mask=None):
    """
    Export TIF files using the provided CRS.
    Експортира TIF файлове с посочената координатна система.
    """
    base_name = f"{os.path.splitext(before_file)[0]}_vs_{os.path.splitext(after_file)[0]}"
    
    if excluded_mask is not None:
        nbr_after_masked = np.ma.masked_where(excluded_mask, nbr_after)
        dNBR_masked = np.ma.masked_where(excluded_mask, dNBR)
        severity_masked = np.ma.masked_where(excluded_mask, burn_severity)
    else:
        nbr_after_masked = nbr_after
        dNBR_masked = dNBR
        severity_masked = burn_severity
    
    files = {}
    
    nbr_path = os.path.join(output_dir, f"{base_name}_nbr_after.tif")
    with rasterio.open(nbr_path, 'w', driver='GTiff', height=nbr_after.shape[0],
                      width=nbr_after.shape[1], count=1, dtype=nbr_after.dtype,
                      crs=crs, transform=transform, compress='lzw') as dst:
        dst.write(nbr_after_masked.filled(np.nan) if isinstance(nbr_after_masked, np.ma.MaskedArray) else nbr_after_masked, 1)
    files['nbr_after'] = nbr_path
    
    dnbr_path = os.path.join(output_dir, f"{base_name}_dnbr.tif")
    with rasterio.open(dnbr_path, 'w', driver='GTiff', height=dNBR.shape[0],
                      width=dNBR.shape[1], count=1, dtype=dNBR.dtype,
                      crs=crs, transform=transform, compress='lzw') as dst:
        dst.write(dNBR_masked.filled(np.nan) if isinstance(dNBR_masked, np.ma.MaskedArray) else dNBR_masked, 1)
    files['dnbr'] = dnbr_path
    
    severity_export = np.full(burn_severity.shape, 255, dtype=np.uint8)
    burned_mask = dNBR >= 0.1
    if excluded_mask is not None:
        burned_mask = burned_mask & (~excluded_mask)
    severity_export[burned_mask] = burn_severity[burned_mask]
    
    severity_path = os.path.join(output_dir, f"{base_name}_burn_severity_3class.tif")
    with rasterio.open(severity_path, 'w', driver='GTiff', height=burn_severity.shape[0],
                      width=burn_severity.shape[1], count=1, dtype=np.uint8,
                      crs=crs, transform=transform, compress='lzw') as dst:
        if excluded_mask is not None:
            severity_export[excluded_mask] = 255
        dst.write(severity_export, 1)
    files['severity'] = severity_path
    
    return files

def add_scale_bar(ax, transform, scale_km=2):
    """
    Add scale bar to plot.
    Добавя линеен мащаб към графиката.
    """
    try:
        pixel_size_m = abs(transform[0])
        scale_length_pixels = (scale_km * 1000) / pixel_size_m
        
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        x_range = xlim[1] - xlim[0]
        y_range = ylim[1] - ylim[0]
        
        x_pos = xlim[1] - x_range * 0.05 - scale_length_pixels
        y_pos = ylim[0] + y_range * 0.05
        
        rect = plt.Rectangle((x_pos, y_pos), scale_length_pixels, y_range * 0.005,
                           facecolor='white', edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        ax.text(x_pos + scale_length_pixels / 2, y_pos - y_range * 0.02,
               f'{scale_km} km', ha='center', va='top', color='white',
               fontweight='bold', fontsize=10,
               bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.7))
    except:
        pass

def create_dashboard(nbr_before, nbr_after, dNBR, burn_severity, overall_stats,
                    before_file, after_file, transform, excluded_mask=None, landcover_raster=None):
    """
    Create dashboard with 3-class burn severity visualization.
    Създава дашборд с визуализация на степен на изгаряне (3 класа).
    """
    if excluded_mask is not None:
        nbr_before_masked = np.ma.masked_where(excluded_mask, nbr_before)
        nbr_after_masked = np.ma.masked_where(excluded_mask, nbr_after)
        dNBR_masked = np.ma.masked_where(excluded_mask, dNBR)
        severity_masked = np.ma.masked_where(excluded_mask, burn_severity)
    else:
        nbr_before_masked = nbr_before
        nbr_after_masked = nbr_after
        dNBR_masked = dNBR
        severity_masked = burn_severity
    
    nbr_before_vis = create_nbr_visualization(nbr_before_masked, excluded_mask, landcover_raster)
    nbr_after_vis = create_nbr_visualization(nbr_after_masked, excluded_mask, landcover_raster)
    dnbr_vis = create_dnbr_visualization(dNBR_masked, excluded_mask, landcover_raster)
    severity_vis = create_severity_visualization(severity_masked, dNBR, excluded_mask, landcover_raster)
    
    fig = plt.figure(figsize=(26, 22), facecolor='white')
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.35, height_ratios=[1, 1, 1, 0.8])
    
    fig.suptitle(f'Burn Severity Analysis (3-Class System)\n{before_file} vs {after_file}', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(nbr_before_vis)
    add_scale_bar(ax1, transform)
    ax1.set_title('NBR - Before Fire', fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(nbr_after_vis)
    add_scale_bar(ax2, transform)
    ax2.set_title('NBR - After Fire', fontsize=12, fontweight='bold')
    ax2.axis('off')
    
    ax3 = fig.add_subplot(gs[0, 2])
    im = ax3.imshow(dnbr_vis)
    add_scale_bar(ax3, transform)
    ax3.set_title('dNBR (Before - After)', fontsize=12, fontweight='bold')
    ax3.axis('off')
    
    norm = mcolors.Normalize(vmin=-0.5, vmax=0.5)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn_r, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax3, fraction=0.046, pad=0.04)
    cbar.set_label('dNBR Value', fontsize=10)
    
    ax4 = fig.add_subplot(gs[1, 0:2])
    ax4.imshow(severity_vis)
    add_scale_bar(ax4, transform)
    ax4.set_title('Burn Severity - 3 Classes', fontsize=12, fontweight='bold')
    ax4.axis('off')
    
    ax_legend = fig.add_subplot(gs[1, 2])
    ax_legend.axis('off')
    
    legend_elements = [
        Patch(facecolor='lightgray', edgecolor='black', label='Unburned/Regrowth (dNBR < 0.1)'),
        Patch(facecolor='yellow', edgecolor='black', label='Low Severity (0.1-0.2)'),
        Patch(facecolor='orange', edgecolor='black', label='Medium Severity (0.2-0.3)'),
        Patch(facecolor='red', edgecolor='black', label='High Severity (>= 0.3)')
    ]
    
    if excluded_mask is not None and landcover_raster is not None:
        legend_elements.append(Patch(facecolor='blue', edgecolor='black', label='Water (excluded)'))
        legend_elements.append(Patch(facecolor='black', edgecolor='white', label='Barren/Urban (excluded)'))
    
    ax_legend.legend(handles=legend_elements, loc='center', fontsize=11, 
                    frameon=True, framealpha=0.9, edgecolor='black',
                    title="Classification Legend", title_fontsize=12)
    
    ax5 = fig.add_subplot(gs[2, 0])
    class_names = ['Low Severity', 'Medium Severity', 'High Severity']
    percentages = [
        overall_stats['burned_stats'][0]['pct_of_burned'],
        overall_stats['burned_stats'][1]['pct_of_burned'],
        overall_stats['burned_stats'][2]['pct_of_burned']
    ]
    colors = ['yellow', 'orange', 'red']
    
    bars = ax5.bar(class_names, percentages, color=colors, edgecolor='black', alpha=0.8, linewidth=2)
    ax5.set_title('Burn Severity Distribution\n(Percentage of Burned Area)', fontsize=12, fontweight='bold')
    ax5.set_ylabel('Percentage of Burned Area (%)', fontsize=11)
    ax5.set_ylim(0, 100)
    ax5.grid(True, alpha=0.3, axis='y')
    ax5.set_xticks(range(len(class_names)))
    ax5.set_xticklabels(class_names, rotation=0, ha='center', fontsize=10)
    
    for bar, pct in zip(bars, percentages):
        if pct > 0:
            ax5.text(bar.get_x() + bar.get_width()/2., pct + 2, f'{pct:.1f}%', 
                    ha='center', va='bottom', fontweight='bold', fontsize=11, color='black')
    
    ax6 = fig.add_subplot(gs[2, 1])
    if excluded_mask is not None:
        dnbr_valid = dNBR[~excluded_mask].flatten()
    else:
        dnbr_valid = dNBR.flatten()
    
    ax6.hist(dnbr_valid, bins=50, color='orange', alpha=0.7, edgecolor='black', linewidth=0.5)
    ax6.axvline(x=0.1, color='gray', linestyle='--', linewidth=2, label='Low threshold (0.1)')
    ax6.axvline(x=0.2, color='orange', linestyle='--', linewidth=2, label='Medium threshold (0.2)')
    ax6.axvline(x=0.3, color='red', linestyle='--', linewidth=2, label='High threshold (0.3)')
    ax6.set_title('dNBR Distribution with Severity Thresholds', fontsize=12, fontweight='bold')
    ax6.set_xlabel('dNBR Value', fontsize=11)
    ax6.set_ylabel('Pixel Count', fontsize=11)
    ax6.grid(True, alpha=0.3)
    ax6.legend(fontsize=10, loc='upper right')
    
    ax7 = fig.add_subplot(gs[3, :])
    ax7.axis('off')
    
    excluded_pixels = np.sum(excluded_mask) if excluded_mask is not None else 0
    total_pixels = burn_severity.size
    excluded_pct = (excluded_pixels / total_pixels) * 100 if total_pixels > 0 else 0
    
    summary_text = [
        "=" * 80,
        "ANALYSIS SUMMARY (3-Class Burn Severity System)",
        "=" * 80,
        "",
        f"Classification Thresholds:",
        f"  • Low Severity:     0.1 <= dNBR < 0.2 (YELLOW)",
        f"  • Medium Severity:  0.2 <= dNBR < 0.3 (ORANGE)",
        f"  • High Severity:    dNBR >= 0.3 (RED)",
        f"  • Unburned/Regrowth: dNBR < 0.1 (LIGHT GRAY)",
        "",
        f"OVERALL STATISTICS (Valid Pixels Only):",
        f"  • Total Valid Pixels: {overall_stats['total_valid']:,}",
        f"  • Unburned Area: {overall_stats['unburned_pct']:.1f}% ({overall_stats['unburned_pixels']:,} pixels)",
        f"  • Total Burned Area: {overall_stats['total_burned_pct']:.1f}% ({overall_stats['total_burned_pixels']:,} pixels)",
        "",
        f"BURN SEVERITY DISTRIBUTION (Percentage of Burned Area):",
        f"  • Low Severity:      {overall_stats['burned_stats'][0]['pct_of_burned']:.1f}% ({overall_stats['burned_stats'][0]['pixels']:,} pixels)",
        f"  • Medium Severity:   {overall_stats['burned_stats'][1]['pct_of_burned']:.1f}% ({overall_stats['burned_stats'][1]['pixels']:,} pixels)",
        f"  • High Severity:     {overall_stats['burned_stats'][2]['pct_of_burned']:.1f}% ({overall_stats['burned_stats'][2]['pixels']:,} pixels)",
        "",
        f"BURN SEVERITY DISTRIBUTION (Percentage of Total Valid Area):",
        f"  • Low Severity:      {overall_stats['burned_stats'][0]['pct_of_total']:.1f}%",
        f"  • Medium Severity:   {overall_stats['burned_stats'][1]['pct_of_total']:.1f}%",
        f"  • High Severity:     {overall_stats['burned_stats'][2]['pct_of_total']:.1f}%",
        "",
        f"Excluded Areas (Water + Barren/Urban):",
        f"  • {excluded_pct:.1f}% of total area ({excluded_pixels:,} pixels)",
        "",
        f"NBR Statistics (Valid Pixels Only):",
        f"  • Pre-fire NBR Mean:  {np.nanmean(nbr_before_masked):.3f}",
        f"  • Post-fire NBR Mean: {np.nanmean(nbr_after_masked):.3f}",
        f"  • dNBR Mean:          {np.nanmean(dNBR_masked):.3f}",
        "",
        "Note: Statistics are calculated on valid pixels only (excluding water and urban areas)"
    ]
    
    ax7.text(0.02, 0.95, "\n".join(summary_text), transform=ax7.transAxes,
             fontsize=10, verticalalignment='top', family='monospace',
             bbox=dict(boxstyle="round,pad=0.5", facecolor='lightgray', alpha=0.8))
    
    return fig

def create_nbr_visualization(nbr, excluded_mask=None, landcover_raster=None):
    if isinstance(nbr, np.ma.MaskedArray):
        nbr_data = nbr.data.copy()
        mask = nbr.mask
    else:
        nbr_data = nbr.copy()
        mask = None
    
    nbr_normalized = (nbr_data + 1) / 2
    nbr_normalized = np.clip(nbr_normalized, 0, 1)
    
    colors = [(0.8, 0, 0), (1.0, 0.5, 0), (1.0, 1.0, 0), (0, 0.8, 0), (0, 0.4, 0)]
    cmap = mcolors.LinearSegmentedColormap.from_list("nbr_cmap", colors)
    
    nbr_colored = (cmap(nbr_normalized) * 255).astype(np.uint8)[:, :, :3]
    
    if excluded_mask is not None:
        if landcover_raster is not None:
            water_mask = excluded_mask & (landcover_raster == 2)
            nbr_colored[water_mask] = [0, 0, 255]
            barren_mask = excluded_mask & (landcover_raster == 3)
            nbr_colored[barren_mask] = [0, 0, 0]
        else:
            nbr_colored[excluded_mask] = [128, 128, 128]
    elif mask is not None:
        nbr_colored[mask] = [128, 128, 128]
    
    return nbr_colored

def create_dnbr_visualization(dNBR, excluded_mask=None, landcover_raster=None):
    if isinstance(dNBR, np.ma.MaskedArray):
        dnbr_data = dNBR.data.copy()
        mask = dNBR.mask
    else:
        dnbr_data = dNBR.copy()
        mask = None
    
    dnbr_normalized = (dnbr_data + 1) / 2
    dnbr_normalized = np.clip(dnbr_normalized, 0, 1)
    
    colors = [(0, 0.6, 0), (0, 0.9, 0), (1.0, 1.0, 0.8), (1.0, 0.6, 0), (0.8, 0, 0)]
    cmap = mcolors.LinearSegmentedColormap.from_list("dnbr_cmap", colors)
    
    dnbr_colored = (cmap(dnbr_normalized) * 255).astype(np.uint8)[:, :, :3]
    
    if excluded_mask is not None:
        if landcover_raster is not None:
            water_mask = excluded_mask & (landcover_raster == 2)
            dnbr_colored[water_mask] = [0, 0, 255]
            barren_mask = excluded_mask & (landcover_raster == 3)
            dnbr_colored[barren_mask] = [0, 0, 0]
        else:
            dnbr_colored[excluded_mask] = [128, 128, 128]
    elif mask is not None:
        dnbr_colored[mask] = [128, 128, 128]
    
    return dnbr_colored

def create_severity_visualization(severity, dNBR, excluded_mask=None, landcover_raster=None):
    height, width = severity.shape
    severity_rgb = np.zeros((height, width, 3), dtype=np.uint8)
    
    burned_mask = dNBR >= 0.1
    if excluded_mask is not None:
        burned_mask = burned_mask & (~excluded_mask)
    
    low_mask = burned_mask & (severity == 0)
    severity_rgb[low_mask] = [255, 255, 0]
    medium_mask = burned_mask & (severity == 1)
    severity_rgb[medium_mask] = [255, 165, 0]
    high_mask = burned_mask & (severity == 2)
    severity_rgb[high_mask] = [255, 0, 0]
    
    if excluded_mask is not None:
        unburned_mask = (~burned_mask) & (~excluded_mask)
    else:
        unburned_mask = ~burned_mask
    severity_rgb[unburned_mask] = [211, 211, 211]
    
    if excluded_mask is not None:
        if landcover_raster is not None:
            water_mask = excluded_mask & (landcover_raster == 2)
            severity_rgb[water_mask] = [0, 0, 255]
            barren_mask = excluded_mask & (landcover_raster == 3)
            severity_rgb[barren_mask] = [0, 0, 0]
        else:
            severity_rgb[excluded_mask] = [128, 128, 128]
    
    return severity_rgb

def save_dashboard(fig, before_file, after_file, output_dir):
    base_name = f"{os.path.splitext(before_file)[0]}_vs_{os.path.splitext(after_file)[0]}"
    dashboard_path = os.path.join(output_dir, f"{base_name}_dashboard.png")
    fig.savefig(dashboard_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  Dashboard saved: {os.path.basename(dashboard_path)}")
    return dashboard_path

def determine_utm_epsg(src):
    left, bottom, right, top = src.bounds
    centre_x = (left + right) / 2
    centre_y = (bottom + top) / 2

    src_crs = src.crs
    dst_crs = CRS.from_epsg(4326)
    transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)
    lon, lat = transformer.transform(centre_x, centre_y)

    if lon < 24.0:
        epsg = 32634   # UTM 34N
    else:
        epsg = 32635   # UTM 35N

    print(f"  Centre longitude: {lon:.4f}°E → selected EPSG:{epsg}")
    return epsg

def process_fire_analysis(before_file, after_file):
    before_path = os.path.join(PREVIEW_DIR, before_file)
    after_path = os.path.join(PREVIEW_DIR, after_file)
    
    try:
        print(f"\n{'='*50}")
        print(f"Processing: {before_file}")
        print(f"       vs: {after_file}")
        print(f"{'='*50}")
        
        with rasterio.open(before_path) as src_before:
            bands_before = [src_before.read(i) for i in range(1, src_before.count + 1)]
            band_info = identify_sentinel2_bands_for_nbr(bands_before)
            if not band_info:
                print("  ERROR: Could not identify bands")
                return None
            nbr_before = calculate_nbr(band_info['nir'], band_info['swir2'])
            transform = src_before.transform
            shape = src_before.shape
            epsg_code = determine_utm_epsg(src_before)
            crs = CRS.from_epsg(epsg_code)
        
        with rasterio.open(after_path) as src_after:
            bands_after = [src_after.read(i) for i in range(1, src_after.count + 1)]
            band_info = identify_sentinel2_bands_for_nbr(bands_after)
            if not band_info:
                print("  ERROR: Could not identify bands")
                return None
            nbr_after = calculate_nbr(band_info['nir'], band_info['swir2'])
        
        dNBR = calculate_dnbr(nbr_before, nbr_after)
        burn_severity_3class = classify_burn_severity_3class(dNBR)
        extended_severity = classify_burn_severity_extended(dNBR)  # 0-4
        
        fire_id = extract_fire_id_from_filename(before_file)
        fire_date = extract_dates_from_filename(after_file)  # using after as fire date
        before_date = extract_dates_from_filename(before_file)
        after_date = extract_dates_from_filename(after_file)
        
        landcover_raster = None
        excluded_mask = None
        
        if fire_id:
            geojson_path = find_landcover_geojson_file(fire_id)
            if geojson_path:
                landcover_raster = load_landcover_geojson(geojson_path, transform, shape)
                if landcover_raster is not None:
                    excluded_mask = create_exclusion_mask(landcover_raster)
        
        overall_stats = calculate_overall_statistics(burn_severity_3class, dNBR, excluded_mask)
        extended_stats = calculate_extended_statistics(extended_severity, landcover_raster, excluded_mask)
        
        # Populate NBR means
        if excluded_mask is not None:
            nbr_before_valid = nbr_before[~excluded_mask]
            nbr_after_valid = nbr_after[~excluded_mask]
            dnbr_valid = dNBR[~excluded_mask]
        else:
            nbr_before_valid = nbr_before.flatten()
            nbr_after_valid = nbr_after.flatten()
            dnbr_valid = dNBR.flatten()
        
        extended_stats['PreFire_NBR_Mean'] = np.nanmean(nbr_before_valid)
        extended_stats['PostFire_NBR_Mean'] = np.nanmean(nbr_after_valid)
        extended_stats['dNBR_Mean'] = np.nanmean(dnbr_valid)
        
        # Build CSV row
        row = {
            'Fire_ID': fire_id,
            'Fire_Date': fire_date,
            'Before_Date': before_date,
            'After_Date': after_date,
            'Before_File': before_file,
            'After_File': after_file,
            'Total_Valid_Pixels': extended_stats['Total_Valid_Pixels'],
            'Excluded_Pixels': extended_stats['Excluded_Pixels'],
            'Excluded_Percentage': extended_stats['Excluded_Percentage'],
            'Total_Burned_Pixels': extended_stats['Total_Burned_Pixels'],
            'Total_Burned_%': extended_stats['Total_Burned_%'],
            'Enhanced_Regrowth_%': extended_stats['Enhanced_Regrowth_%'],
            'Unburned_%': extended_stats['Unburned_%'],
            'Low_Severity_%': extended_stats['Low_Severity_%'],
            'Moderate_Severity_%': extended_stats['Moderate_Severity_%'],
            'High_Severity_%': extended_stats['High_Severity_%'],
            'Most_Common_Severity': extended_stats['Most_Common_Severity'],
            'Fire_Impact_Assessment': extended_stats['Fire_Impact_Assessment'],
            'Damage_Assessment': extended_stats['Damage_Assessment'],
            'PreFire_NBR_Mean': extended_stats['PreFire_NBR_Mean'],
            'PostFire_NBR_Mean': extended_stats['PostFire_NBR_Mean'],
            'dNBR_Mean': extended_stats['dNBR_Mean'],
            'Has_LandCover_Data': extended_stats['Has_LandCover_Data'],
            'Dominant_LandCover': extended_stats['Dominant_LandCover'],
            'Dominant_LandCover_%': extended_stats['Dominant_LandCover_%'],
            'Most_Vulnerable_HighSeverity': extended_stats.get('Most_Vulnerable_HighSeverity', 'N/A'),
            'Most_Vulnerable_HighSeverity_%': extended_stats.get('Most_Vulnerable_HighSeverity_%', np.nan),
            'Most_Vulnerable_ModerateSeverity': extended_stats.get('Most_Vulnerable_ModerateSeverity', 'N/A'),
            'Most_Vulnerable_ModerateSeverity_%': extended_stats.get('Most_Vulnerable_ModerateSeverity_%', np.nan),
            'Most_Vulnerable_LowSeverity': extended_stats.get('Most_Vulnerable_LowSeverity', 'N/A'),
            'Most_Vulnerable_LowSeverity_%': extended_stats.get('Most_Vulnerable_LowSeverity_%', np.nan),
        }
        
        # Landcover‑specific columns
        for class_id in LANDCOVER_CLASSES:
            prefix = LANDCOVER_CLASSES[class_id]['name'].replace(' ', '_')
            lc = extended_stats['landcover_stats'].get(class_id, {'present': False})
            if lc['present']:
                row[f'{prefix}_Pixels'] = lc['pixels']
                row[f'{prefix}_%_of_Total'] = lc['percent_of_total']
                row[f'{prefix}_EnhancedRegrowth_%'] = lc['regrowth_pct']
                row[f'{prefix}_Unburned_%'] = lc['unburned_pct']
                row[f'{prefix}_LowSeverity_%'] = lc['low_pct']
                row[f'{prefix}_ModerateSeverity_%'] = lc['moderate_pct']
                row[f'{prefix}_HighSeverity_%'] = lc['high_pct']
                row[f'{prefix}_ModerateHigh_%'] = lc['moderate_high_pct']
                row[f'{prefix}_LowModerateHigh_%'] = lc['low_mod_high_pct']
                row[f'{prefix}_Vulnerability_Index'] = lc['vulnerability_index']
            else:
                row[f'{prefix}_Pixels'] = 0
                row[f'{prefix}_%_of_Total'] = 0
                row[f'{prefix}_EnhancedRegrowth_%'] = 0
                row[f'{prefix}_Unburned_%'] = 0
                row[f'{prefix}_LowSeverity_%'] = 0
                row[f'{prefix}_ModerateSeverity_%'] = 0
                row[f'{prefix}_HighSeverity_%'] = 0
                row[f'{prefix}_ModerateHigh_%'] = 0
                row[f'{prefix}_LowModerateHigh_%'] = 0
                row[f'{prefix}_Vulnerability_Index'] = 0
        
        # Export TIF files
        print("\nExporting TIF files...")
        tif_files = export_tif_files(nbr_after, dNBR, burn_severity_3class, transform,
                                    before_file, after_file, NBR_DIR, crs, excluded_mask)
        
        print("\nCreating dashboard...")
        fig = create_dashboard(nbr_before, nbr_after, dNBR, burn_severity_3class, overall_stats,
                              before_file, after_file, transform, excluded_mask, landcover_raster)
        dashboard_path = save_dashboard(fig, before_file, after_file, NBR_DIR)
        display_dashboard(dashboard_path, fire_id)
        
        print(f"\n{'='*50}")
        print("RESULTS SUMMARY:")
        print(f"{'='*50}")
        print(f"Fire ID: {fire_id}")
        print(f"Total Burned: {extended_stats['Total_Burned_%']:.1f}%")
        print(f"Low / Moderate / High: {extended_stats['Low_Severity_%']:.1f}% / {extended_stats['Moderate_Severity_%']:.1f}% / {extended_stats['High_Severity_%']:.1f}%")
        
        return {
            'fire_id': fire_id,
            'overall_stats': overall_stats,
            'extended_stats': row,  # CSV row
            'tif_files': tif_files,
            'dashboard': dashboard_path
        }
        
    except Exception as e:
        print(f"ERROR processing {before_file}: {e}")
        import traceback
        traceback.print_exc()
        return None

def export_stats_to_csv(results_list, output_dir):
    """
    Create a DataFrame from all fire results and save to CSV.
    Създава DataFrame от резултатите за всички пожари и записва в CSV.
    """
    rows = []
    for res in results_list:
        if res and 'extended_stats' in res:
            rows.append(res['extended_stats'])
    
    if not rows:
        print("No results to export.")
        return
    
    df = pd.DataFrame(rows)
    # Reorder columns to match the desired layout (approximate)
    col_order = [
        'Fire_ID', 'Fire_Date', 'Before_Date', 'After_Date', 'Before_File', 'After_File',
        'Total_Valid_Pixels', 'Excluded_Pixels', 'Excluded_Percentage',
        'Total_Burned_Pixels', 'Total_Burned_%', 'Enhanced_Regrowth_%', 'Unburned_%',
        'Low_Severity_%', 'Moderate_Severity_%', 'High_Severity_%',
        'Most_Common_Severity', 'Fire_Impact_Assessment', 'Damage_Assessment',
        'PreFire_NBR_Mean', 'PostFire_NBR_Mean', 'dNBR_Mean',
        'Has_LandCover_Data', 'Dominant_LandCover', 'Dominant_LandCover_%',
        'Most_Vulnerable_HighSeverity', 'Most_Vulnerable_HighSeverity_%',
        'Most_Vulnerable_ModerateSeverity', 'Most_Vulnerable_ModerateSeverity_%',
        'Most_Vulnerable_LowSeverity', 'Most_Vulnerable_LowSeverity_%',
        'Field/Agriculture_Pixels', 'Field/Agriculture_%_of_Total',
        'Field/Agriculture_EnhancedRegrowth_%', 'Field/Agriculture_Unburned_%',
        'Field/Agriculture_LowSeverity_%', 'Field/Agriculture_ModerateSeverity_%',
        'Field/Agriculture_HighSeverity_%', 'Field/Agriculture_ModerateHigh_%',
        'Field/Agriculture_LowModerateHigh_%', 'Field/Agriculture_Vulnerability_Index',
        'Coniferous_Forest_Pixels', 'Coniferous_Forest_%_of_Total',
        'Coniferous_Forest_EnhancedRegrowth_%', 'Coniferous_Forest_Unburned_%',
        'Coniferous_Forest_LowSeverity_%', 'Coniferous_Forest_ModerateSeverity_%',
        'Coniferous_Forest_HighSeverity_%', 'Coniferous_Forest_ModerateHigh_%',
        'Coniferous_Forest_LowModerateHigh_%', 'Coniferous_Forest_Vulnerability_Index',
        'Deciduous_Forest_Pixels', 'Deciduous_Forest_%_of_Total',
        'Deciduous_Forest_EnhancedRegrowth_%', 'Deciduous_Forest_Unburned_%',
        'Deciduous_Forest_LowSeverity_%', 'Deciduous_Forest_ModerateSeverity_%',
        'Deciduous_Forest_HighSeverity_%', 'Deciduous_Forest_ModerateHigh_%',
        'Deciduous_Forest_LowModerateHigh_%', 'Deciduous_Forest_Vulnerability_Index',
        'Water_Pixels', 'Water_%_of_Total',
        'Water_EnhancedRegrowth_%', 'Water_Unburned_%',
        'Water_LowSeverity_%', 'Water_ModerateSeverity_%', 'Water_HighSeverity_%',
        'Bare_and_Urban_Territories_Pixels', 'Bare_and_Urban_Territories_%_of_Total',
        'Bare_and_Urban_Territories_EnhancedRegrowth_%',
        'Bare_and_Urban_Territories_Unburned_%',
        'Bare_and_Urban_Territories_LowSeverity_%',
        'Bare_and_Urban_Territories_ModerateSeverity_%',
        'Bare_and_Urban_Territories_HighSeverity_%'
    ]
    
    # Ensure all columns exist, fill missing with N/A
    for col in col_order:
        if col not in df.columns:
            df[col] = np.nan
    df = df[col_order]
    
    csv_path = os.path.join(output_dir, 'fire_statistics_summary.csv')
    df.to_csv(csv_path, index=False, na_rep='N/A')
    print(f"\nFull statistics CSV exported to: {csv_path}")
    return csv_path

def main():
    print("="*80)
    print("BURN SEVERITY ANALYSIS (3‑Class + Regrowth) WITH DETAILED CSV EXPORT")
    print("="*80)
    
    _, fire_groups = find_and_group_image_files()
    
    if not fire_groups:
        print("\nNo fire groups found. Please check PREVIEW_DIR path.")
        return
    
    results = []
    
    # Process all fires (skip none, to have full CSV)
    for fire_id in sorted(fire_groups.keys(), key=lambda x: int(x)):
        if fire_groups[fire_id]['before'] and fire_groups[fire_id]['after']:
            before_file = sorted(fire_groups[fire_id]['before'])[0]
            after_file = sorted(fire_groups[fire_id]['after'])[0]
            res = process_fire_analysis(before_file, after_file)
            if res:
                results.append(res)
    
    # Export CSV
    export_stats_to_csv(results, NBR_DIR)
    
    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)

if __name__ == "__main__":
    main()